In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from network.model import NeuralNetwork
import cv2

In [2]:
# Configurazione paths
BASE_PATH = "/Users/edoardoconti/phd_local/projects/iclus_ordinal/risultati/results_ordinal_paper"
MODEL_PATHS = [
    "seed94_clm_resnet50_100_[32]_[0.3]_['cloglog']_[True]_QWK_SDG_[0.01]/fold_{fold}/holdout_{holdout}/weights/best_weights.h5",
    "seed94_cnnregstn_100_[32]_[0.3]_SORD_Adam_[1e-4]/fold_{fold}/holdout_{holdout}/weights/best_weights.h5",
    "seed94_obd_resnet50_100_[32]_[0.3]_[8192]_ODL_SDG_['cdr']/fold_{fold}/holdout_{holdout}/weights/best_weights.h5",
    "seed94_resnet50_100_[32]_[0.3]_CCE_SDG_['cdr']/fold_{fold}/holdout_{holdout}/weights/best_weights.h5",
    "seed94_resnet50_100_[32]_[0.3]_QWK_SDG_['cdr']/fold_{fold}/holdout_{holdout}/weights/best_weights.h5",
]
IMAGE_DIR = "/Users/edoardoconti/Downloads/Images_for_Marche"
OUTPUT_DIR = "output_gradcams"

In [3]:
def extract_config_from_path(model_path):
    """Estrai parametri di configurazione dal path del modello"""
    config = {
        'ds_img_size': 224,
        'ds_num_classes': 4,
        'nn_backbone': 'resnet50',
        'nn_dropout': 0.3,
        'clm_link': None,
        'obd_hidden_size': None
    }
    
    if 'clm' in model_path.lower():
        config['clm_link'] = 'cloglog'
    elif 'obd' in model_path.lower():
        config['obd_hidden_size'] = 8192
        
    # Estrai dropout se presente nel path
    if '_[' in model_path and ']_' in model_path:
        try:
            config['nn_dropout'] = float(model_path.split('_[')[2].split(']_')[0])
        except:
            pass
            
    return config

def extract_loss_from_path(model_path):
    """Estrai la metrica dal path del modello"""
    metrics = ['QWK', 'SORD', 'ODL', 'CCE']
    for metric in metrics:
        if metric in model_path:
            return metric
    return None

In [4]:
def find_layer_recursive(model, layer_name):
    """Trova un layer ricorsivamente nel modello"""
    for layer in model.layers:
        if layer.name == layer_name:
            print(f"Trovato layer {layer.name} - Tipo: {type(layer)}")
            return layer
        if hasattr(layer, 'layers'):
            found_layer = find_layer_recursive(layer, layer_name)
            if found_layer:
                return found_layer
    raise ValueError(f"Layer {layer_name} non trovato")

In [5]:
def build_model_from_path(model_path):
    """Costruisci modello in base al tipo specificato nel path"""
    config = extract_config_from_path(model_path)
    builder = NeuralNetwork(**config)
    
    if 'clm' in model_path.lower():
        return builder.clm()
    elif 'obd' in model_path.lower():
        return builder.obd()
    elif 'cnnregstn' in model_path.lower():
        return builder.cnnstn()
    else:
        return builder.resnet50()

In [6]:
def find_last_conv_layer(model):
    """Trova l'ultimo layer convoluzionale nel modello"""
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Convolution2D)):
            print(f"Selezionato layer convoluzionale: {layer.name}")
            return layer.name
    raise ValueError("Nessun layer convoluzionale trovato")

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Genera la mappa di attivazione GradCAM"""
    if 'cnnstn' in model.name.lower():
        last_conv_layer = find_layer_recursive(model, last_conv_layer_name)
    else:
        last_conv_layer = model.get_layer(last_conv_layer_name)

    grad_model = Model(model.inputs, [last_conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        
        pred_index = pred_index or tf.argmax(preds[0])
        pred_index -= tf.cast('obd' in model.name.lower() and pred_index == 3, tf.int64)

        # if pred_index is None:
        #     if 'obd' in model.name.lower():
        #         preds = tf.sigmoid(preds)  # Converti logit in probabilità per OBD
        #         pred_index = tf.cast(tf.reduce_sum(tf.cast(preds > 0.5, tf.int32), axis=1), tf.int32) - 1
        #     else:
        #         pred_index = tf.argmax(preds[0])
    
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    print(f"Heatmap stats - Min: {np.min(heatmap):.4f}, Max: {np.max(heatmap):.4f}, Mean: {np.mean(heatmap):.4f}")
    return heatmap.numpy()

In [8]:
def merge_gradcam(img, heatmap, alpha=0.5):
    """Fonde l'immagine originale con la heatmap"""
    img = image.img_to_array(img)
    img = img / 255.0  # Normalizza l'immagine
    
    # Resize heatmap per matchare le dimensioni dell'immagine
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * np.nan_to_num(heatmap, nan=0))
    
    jet = matplotlib.colormaps.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    
    # Normalizza la heatmap
    jet_heatmap = jet_heatmap / np.max(jet_heatmap)
    
    # Sovrapponi heatmap all'immagine
    superimposed_img = jet_heatmap * alpha + img * (1 - alpha)
    superimposed_img = image.array_to_img(superimposed_img)
    
    return superimposed_img

In [9]:
def process_images_for_model(model_path, output_subdir):
    """Elabora tutte le immagini per un singolo modello"""
    try:
        model = build_model_from_path(model_path)
        model.load_weights(model_path)
        print(f"\nModello {os.path.basename(model_path)} caricato correttamente")
    except Exception as e:
        print(f"Errore nel caricamento del modello {os.path.basename(model_path)}: {str(e)}")
        return
        
    last_conv_layer_name = find_last_conv_layer(model)
    
    for score in ['0', '1', '2', '3']:
        score_dir = os.path.join(IMAGE_DIR, score)
        output_score_dir = os.path.join(output_subdir, score)
        os.makedirs(output_score_dir, exist_ok=True)
        
        for img_name in os.listdir(score_dir):
            if not img_name.endswith('.png'):
                continue
                
            img_path = os.path.join(score_dir, img_name)
            img = image.load_img(img_path, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            #img_array = preprocess_input(img_array)
            
            print(f"\nElaborazione immagine {img_name} (score {score})")
            heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, int(score))
            
            # Genera e salva immagine con GradCAM sovrapposto
            superimposed_img = merge_gradcam(img, heatmap)
            output_path = os.path.join(output_score_dir, f"gradcam_{img_name}")
            superimposed_img.save(output_path)
            print(f"Salvato GradCAM in {output_path}")

In [10]:
# Elabora tutti i modelli
for model_idx, model_template in enumerate(MODEL_PATHS):
    model_arch = model_template.split('_')[1]  # es. 'clm', 'obd'
    model_loss = extract_loss_from_path(model_template)
    model_name = f"{model_arch}_{model_loss}"
    
    # Crea cartella principale per il modello
    model_main_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_main_dir, exist_ok=True)

    for fold in [1, 2, 3, 4, 5]:
        for holdout in [1, 2, 3]:
            model_path = os.path.join(BASE_PATH, model_template.format(fold=fold, holdout=holdout))
            if not os.path.exists(model_path):
                print(f"File non trovato: {model_path}")
                continue
                
            output_subdir = os.path.join(model_main_dir, f"fold{fold}_holdout{holdout}")
            print(f"{'='*100}\nElaborazione modello {model_name} (fold {fold}, holdout {holdout})\n{'='*100}")
            process_images_for_model(model_path, output_subdir)

Elaborazione modello clm_QWK (fold 1, holdout 1)


2025-04-28 15:15:38.036344: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-04-28 15:15:38.036365: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-04-28 15:15:38.036368: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-04-28 15:15:38.036532: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-28 15:15:38.036705: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



Modello best_weights.h5 caricato correttamente
Selezionato layer convoluzionale: conv5_block3_3_conv

Elaborazione immagine convex_20200319112643_1137450.mat_frame_82.png (score 0)
Heatmap stats - Min: 0.0000, Max: 1.0000, Mean: 0.3094
Salvato GradCAM in output_gradcams/clm_QWK/fold1_holdout1/0/gradcam_convex_20200319112643_1137450.mat_frame_82.png

Elaborazione immagine convex_3.7.mat_frame_15.png (score 0)
Heatmap stats - Min: 0.0000, Max: 1.0000, Mean: 0.1732
Salvato GradCAM in output_gradcams/clm_QWK/fold1_holdout1/0/gradcam_convex_3.7.mat_frame_15.png

Elaborazione immagine convex_20200319112643_1138510.mat_frame_205.png (score 0)
Heatmap stats - Min: 0.0000, Max: 1.0000, Mean: 0.2744
Salvato GradCAM in output_gradcams/clm_QWK/fold1_holdout1/0/gradcam_convex_20200319112643_1138510.mat_frame_205.png

Elaborazione immagine convex_202003131532410435ABD.mat_frame_54.png (score 1)
Heatmap stats - Min: 0.0000, Max: 1.0000, Mean: 0.2154
Salvato GradCAM in output_gradcams/clm_QWK/fold1_h

InvalidArgumentError: {{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:GPU:0}} slice index 3 of dimension 1 out of bounds. [Op:StridedSlice] name: strided_slice/

In [ ]:
import os
import shutil

# Define source and destination paths
src_dir = OUTPUT_DIR
dst_dir = os.path.expanduser("~/Downloads/GRADCAMS_OUT_4")

# Process each model's output
for model_folder in os.listdir(src_dir):
    if not os.path.isdir(os.path.join(src_dir, model_folder)):
        continue
        
    model_name = model_folder  # e.g. 'clm_QWK'
    
    for fold_holdout in os.listdir(os.path.join(src_dir, model_folder)):
        fold, holdout = fold_holdout.split('_')
        fold_num = fold.replace('fold', '')
        holdout_num = holdout.replace('holdout', '')
        
        for score in ['0', '1', '2', '3']:
            src_score_dir = os.path.join(src_dir, model_folder, fold_holdout, score)
            if not os.path.exists(src_score_dir):
                continue
                
            for img_file in os.listdir(src_score_dir):
                if img_file.endswith('.png'):
                    # Create subfolder for each input image
                    base_name = os.path.splitext(img_file.replace('gradcam_', ''))[0]
                    img_subdir = os.path.join(dst_dir, score, base_name)
                    os.makedirs(img_subdir, exist_ok=True)
                    
                    # Create new filename with model info
                    new_name = f"{model_name}_FOLD{fold_num}_HOLDOUT{holdout_num}.png"
                    dst_path = os.path.join(img_subdir, new_name)
                    
                    # Copy file
                    shutil.copy2(os.path.join(src_score_dir, img_file), dst_path)
                    print(f"Copied {img_file} to {dst_path}")